# Sensibilidade historica a divisao pelo P95

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/15bMoG0kMxF8at3fLT_ga81FC6_MwUWd_

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. 

# Etapa 06 - Sensibilidade dos agrupamentos ao P95

**Pergunta:** quanto a organizacao dos grupos muda quando omitimos somente a divisao final pelo P95?

Esta comparacao foi motivada pela auditoria DMSO da etapa 05. E uma analise **pos-hoc e condicional**, nao validacao independente nem busca de uma representacao vencedora.

Comparamos **A = RMS/P95** com **E = RMS ja normalizado por MAD**, mantendo as 2.447 condicoes completas, os quatro canais e **k fixo: 2 para magnitude e 4 para contrastes**. Os modelos, as 12 prioridades e a avaliacao original de R4 permanecem preservados. Nao lemos R4.

Nao usar `E > 1` como limiar de atividade. Os novos grupos H sao apenas diagnosticos de sensibilidade; nao sao equivalentes aos grupos G nem novos mecanismos biologicos.

O notebook e autonomo: abra no Colab, autorize o Drive e execute todas as celulas. Nao e preciso executar novamente os notebooks anteriores. A versao entregue nao contem saidas fabricadas; a verificacao local fica separada no pacote.


In [ ]:
from pathlib import Path
import os, sys, importlib.util, subprocess
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
required=[('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]
if IN_COLAB:
    required.append(('polars','polars'))
for module,package in required:
    if importlib.util.find_spec(module) is None:
        if not IN_COLAB: raise ImportError('Dependencia ausente: '+package)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
from IPython.display import display, Markdown, Image
RAIZ=Path(os.environ.get('TCC_RAIZ','/content/drive/MyDrive/mba_tcc'))
print('Raiz:', RAIZ)


## 1. Fontes, integridade e protocolo fixo
Os CSV sao lidos com Polars quando disponivel. Pandas e alternativa explicita registrada. Nos tratamentos, E e recuperado como A vezes o P95 fixo da mesma placa; a constancia desse denominador permite recuperar tambem as medianas. Nenhuma nova normalizacao por tratamento e introduzida.

In [ ]:
"""Etapa 06: sensibilidade exploratoria a divisao por P95.
Nao substitui os modelos originais; k fixo, sem R4, sem novos hits.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, shutil
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt

P,R,W,D,E='Metadata_Plate_ID','Metadata_Replicate','Metadata_Well','Metadata_Concentration','Metadata_EOS'
KEY=[P,E,'_dose_key']
CHANNELS=['AGP','DNA','ER','Mito']
BLOCKS=CHANNELS+['Morfologia_Cells','Morfologia_Cyto','Morfologia_Nuc']
REPS=['R1','R2','R3']; PLATES=['B100'+str(i) for i in range(1,8)]
SPACES={'magnitude_4canais':2,'contrastes_4canais':4}
CFG={'seed':20260926,'n_init':30,'max_iter':500,'n_subsamples':20,'fraction':0.8,'k_fixed':SPACES}
SOURCES = {
  "complete": {
    "relative_path": "resultados_tcc/mapa_multibloco/R1_R3_20260926T152153_537184Z/tabelas/completas.csv",
    "sha256": "de4566acaa64208b5e41294ac9e4acc506833a23541ae65f5c0f383fa05bddcf",
    "drive_url": "https://drive.google.com/file/d/1vB1LoU1oZQg1CxjYqKbRClxhfsNblU2P/view"
  },
  "refs": {
    "relative_path": "pipeline_IMTM_HepG2/20260919T204105_625149Z/normalizacao_20260919T205516_579360Z/feature_selection_20260919T211235_656816Z/escores_exploratorios_20260919T230543_380473Z/referencias_dmso.csv",
    "sha256": "05ea6446566459559a376078d14c7ec5ef98821e4d34bab6c788ac8ebed6d952",
    "drive_url": "https://drive.google.com/file/d/1SgytsR_sz-st5I7Q0ive0SewOZmh5yKx/view"
  },
  "dmso": {
    "relative_path": "resultados_tcc/auditoria_DMSO_P95/R1_R3_20260927T123126_778389Z/tabelas/DMSO_escores_e_contrastes.csv",
    "sha256": "4721bd5db4f9a9dd820438d940abbf0ba0622e0e32e7f1977b4f25cf17f9de43",
    "drive_url": "https://drive.google.com/file/d/19CUP_xj1M0Cspe2KkL_lXr7zZ0mdko87/view"
  },
  "models": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/modelos_numericos.json",
    "sha256": "5839e78093939ff2c89c6f329ae41588023ad2121f53f8909905a59100656985",
    "drive_url": "https://drive.google.com/file/d/1GDmrHEMOFWuZPj4Zt57qVU7qqNTKGK63/view"
  },
  "groups": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/grupos.csv",
    "sha256": "801a90adf37da01108aec71acae2c9f78c5c3552248faadb61fdf2e3398c6019",
    "drive_url": "https://drive.google.com/file/d/1wf33vAKy6IBYswYG1t7e8HMFkmpdI1eA/view"
  },
  "assign": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/atribuicoes.csv",
    "sha256": "27a1aaff1046de55e0721f62e1cfe166c83300df7192785e02e5616c893c43c4",
    "drive_url": "https://drive.google.com/file/d/19-cPZoxd0JTW6jOLP_IZ4Md9cVsWJZys/view"
  },
  "candidates": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/candidatos.csv",
    "sha256": "4717be42bbf400925149d07339036735474575ec37e4b766ee5b8dc5a0df8ca7",
    "drive_url": "https://drive.google.com/file/d/1zhBCydnXSkOH2lMU2tqCczXr1_BfslDn/view"
  },
  "subs": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/subamostras.csv",
    "sha256": "43b932704416ca8cbe0afa1387bdb68813492a8e8fcf3a421afbc987d6c4e46f",
    "drive_url": "https://drive.google.com/file/d/1Kjzgx_M1hibP8FRO4U6ybONxfvJnKd0e/view"
  },
  "rot": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/rotacoes.csv",
    "sha256": "d5a9cb08b0826a08588c7c5d7c5cd3c4ac7b225d0009feaadecda9e0360a55ee",
    "drive_url": "https://drive.google.com/file/d/145RsDiUYLFZey92LdmOBe8MJh489kWka/view"
  }
}


def check(ok,message):
    if not bool(ok): raise ValueError(message)


def close(a,b,message):
    check(np.allclose(a,b,rtol=1e-9,atol=1e-10,equal_nan=False),message)


def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()


def read_csv(path):
    """Polars quando disponivel; pandas como alternativa explicita para CSV."""
    try: import polars as pl
    except ImportError: return pd.read_csv(path,dtype={D:str}),'pandas'
    import csv
    with Path(path).open(encoding='utf-8-sig',newline='') as f: header=next(csv.reader(f))
    overrides={D:pl.String} if D in header else None
    z=pl.read_csv(path,infer_schema_length=10000,schema_overrides=overrides)
    return pd.DataFrame(z.to_dict(as_series=False)),'polars'


def bool_col(s):
    z=s.astype(str).str.lower().str.strip().map({'true':True,'false':False,'1':True,'0':False})
    check(z.notna().all(),'Booleano invalido.');return z.astype(bool)


def rep(x,space):
    x=np.asarray(x,float)
    check(x.ndim==2 and x.shape[1]==4 and np.isfinite(x).all() and (x>=0).all(),
          'Representacao exige quatro canais finitos e nao negativos.')
    l=np.log1p(x)
    if space=='contrastes_4canais': return l-l.mean(axis=1,keepdims=True)
    check(space=='magnitude_4canais','Espaco desconhecido.');return l


def fixed_predict(x,m):
    base=rep(x,m['espaco']);sc=np.asarray(m['scale'],float)
    check(np.all(sc>0) and m['channels']==CHANNELS,'Modelo salvo invalido.')
    z=(base-np.asarray(m['center']))/sc
    d=((z[:,None,:]-np.asarray(m['cluster_centers'])[None,:,:])**2).sum(axis=2)
    return d.argmin(axis=1),z


def fit(x,space,k,seed):
    base=rep(x,space)
    check(np.unique(base,axis=0).shape[0]>=k,'Numero insuficiente de perfis distintos.')
    sc=RobustScaler(quantile_range=(25,75));z=sc.fit_transform(base)
    km=KMeans(n_clusters=k,init='k-means++',n_init=CFG['n_init'],max_iter=CFG['max_iter'],
              random_state=seed,algorithm='lloyd').fit(z)
    check(len(np.unique(km.labels_))==k,'Agrupamento degenerado.')
    m={'espaco':space,'k':k,'channels':CHANNELS,'center':sc.center_.tolist(),
       'scale':sc.scale_.tolist(),'cluster_centers':km.cluster_centers_.tolist(),'seed':seed}
    y,_=fixed_predict(x,m);check(np.array_equal(y,km.labels_),'Projecao nao reproduz ajuste.')
    return m,y,z


def load(root):
    frames={};paths={};readers={}
    for name,s in SOURCES.items():
        p=Path(root)/s['relative_path'];check(p.is_file(),'Fonte ausente: '+str(p))
        check(sha256(p)==s['sha256'],'Integridade alterada: '+str(p))
        paths[name]=p
        if name=='models':frames[name]=json.loads(p.read_text())
        else:frames[name],readers[name]=read_csv(p)
    df=frames['complete'].copy();at=frames['assign'].copy();dm=frames['dmso'].copy()
    for z in (df,at):
        check(not z[KEY].isna().any().any(),'Chave nula.')
        for c in KEY:z[c]=z[c].astype(str).str.strip();check(z[c].ne('').all(),'Chave vazia.')
        check(not z.duplicated(KEY).any() and len(z)==2447,'Condicoes duplicadas ou numero diferente.')
        check(not any('R4' in c for c in z),'Fonte contem coluna R4.')
    df=df.sort_values(KEY,kind='stable').reset_index(drop=True)
    at=at.sort_values(KEY,kind='stable').reset_index(drop=True)
    check(df[KEY].equals(at[KEY]),'Chaves das atribuicoes divergem.')
    check(bool_col(df.completa_3rep_7blocos).all(),'Condicoes incompletas; nao ha imputacao.')
    df['prioridade_R1_R3_original']=bool_col(df.prioridade_R1_R3_original)
    check(df.prioridade_R1_R3_original.sum()==12,'Prioridades originais divergiram.')
    check(set(df[P])==set(PLATES),'Placas diferentes.')
    refs=frames['refs'].query('bloco in @BLOCKS').copy()
    check(len(refs)==49 and not refs.duplicated([P,'bloco']).any(),'Referencias ausentes/duplicadas.')
    check((refs.n_dmso_validos==84).all() and np.isclose(refs.quantil,.95).all(),'P95 com amostragem diferente.')
    q=refs.pivot(index=P,columns='bloco',values='limiar_RMS').loc[PLATES,BLOCKS]
    check(np.isfinite(q).all().all() and (q>0).all().all(),'P95 nao finito/positivo.')
    a=np.stack([df[['A_'+b+'_'+r for b in BLOCKS]].to_numpy(float) for r in REPS],axis=1)
    check(np.isfinite(a).all() and (a>=0).all(),'Escores nao finitos/negativos.')
    close(np.median(a,axis=1),df[['A_'+b+'_mediana' for b in BLOCKS]],'Medianas nao reproduzidas.')
    # P95 e constante por placa/bloco: mediana(A*P95)=mediana(A)*P95.
    # A auditoria 05 conferiu esta identidade nos arquivos de pocos originais.
    qrows=q.loc[df[P],BLOCKS].to_numpy(float)
    ecube=a*qrows[:,None,:]
    close(ecube/qrows[:,None,:],a,'Falha na identidade E=A*P95.')
    check(len(dm)==588 and not dm.duplicated([P,R,W]).any(),'DMSO duplicados ou numero incorreto.')
    check(set(dm[R])==set(REPS) and set(dm[P])==set(PLATES),'DMSO fora do escopo.')
    counts=dm.groupby([P,R]).size()
    check(len(counts)==21 and (counts==28).all(),'DMSO: esperados 28 por placa e replica.')
    av=dm[['A_'+b for b in CHANNELS]].to_numpy(float)
    ev=dm[['E_'+b for b in CHANNELS]].to_numpy(float)
    close(ev/q.loc[dm[P],CHANNELS].to_numpy(),av,'DMSO: A diferente de E/P95.')
    check(np.isfinite(ev).all() and (ev>=0).all(),'RMS dos controles invalido.')
    for pref,raw in [('C_A_',av),('C_E_',ev)]:
        close(rep(raw,'contrastes_4canais'),dm[[pref+b for b in CHANNELS]],'Contrastes DMSO diferentes.')
    frozen={};maps={};conferences=[]
    for space,k in SPACES.items():
        gs=frames['groups'].query('espaco == @space');check(set(gs.k)=={k},'k original diferente.')
        mm=[m for m in frames['models'] if m['espaco']==space and m['k']==k]
        check(len(mm)==1,'Modelo original ausente/ambiguo.');m=mm[0]
        mapping=dict(zip(gs.id_modelo.astype(int),gs.grupo.astype(str)))
        for r,raw in [('mediana',np.median(a[:,:,:4],axis=1))]+[(r,a[:,i,:4]) for i,r in enumerate(REPS)]:
            y,_=fixed_predict(raw,m);lab=np.array([mapping[int(g)] for g in y])
            col=space+('_grupo' if r=='mediana' else '_'+r)
            check(np.array_equal(lab,at[col]),'Modelo original nao reproduz '+col)
            conferences.append({'espaco':space,'projecao':r,'n':len(y),'n_divergencias':0})
        check(len(frames['rot'].query('espaco == @space and k == @k'))==3,'Rotacoes originais ausentes.')
        check(len(frames['subs'].query('espaco == @space and k == @k'))==20,'Subamostras originais ausentes.')
        frozen[space]=m;maps[space]=mapping
    return {'df':df,'A':a,'E':ecube,'dmso':dm,'frozen':frozen,'maps':maps,'frames':frames,
            'paths':paths,'readers':readers,'conferences':pd.DataFrame(conferences)}


def group_masks(comp):
    unique=np.array(sorted(set(comp)));rng=np.random.default_rng(CFG['seed'])
    n=min(len(unique)-1,max(2,int(np.ceil(CFG['fraction']*len(unique)))))
    return [np.isin(comp,rng.choice(unique,n,replace=False)) for _ in range(CFG['n_subsamples'])]


def eta2(x,labels):
    x=np.asarray(x,float);tot=np.sum((x-x.mean())**2)
    if tot==0:return np.nan
    between=sum(np.sum(labels==g)*(x[labels==g].mean()-x.mean())**2 for g in np.unique(labels))
    return float(between/tot)




## 2. Analise pareada
A referencia A usa os modelos e tabelas de estabilidade originais. A alternativa E usa o mesmo k, as mesmas sementes e os mesmos sorteios de compostos da etapa 04. Reajustam-se a escala e o K-means somente na nova representacao e nas suas repeticoes. Nao se escolhe outro k depois de olhar os resultados.

In [ ]:
def analyse(inp,out):
    df,dm=inp['df'],inp['dmso'];ca,ce=inp['A'],inp['E']
    med={'A':np.median(ca,axis=1),'E':np.median(ce,axis=1)}
    masks=group_masks(df[E].to_numpy());plate=df[P].to_numpy()
    summaries=[];subs=[];rots=[];profiles=[];composition=[];cross=[];effect=[];dmrows=[];models=[]
    assignments=df[KEY+[D,'prioridade_R1_R3_original']].copy()
    with threadpool_limits(limits=1):
        for space,k in SPACES.items():
            original=inp['frozen'][space];seed=int(original['seed'])
            model_e,ye,ze=fit(med['E'][:,:4],space,k,seed)
            models.append(dict(model_e,entrada='E_RMS_pos_MAD_sem_divisao_P95'))
            ya,za=fixed_predict(med['A'][:,:4],original)
            order=sorted(np.unique(ye),key=lambda g:(np.median(np.log1p(med['E'][ye==g,:4]).mean(1)),int(g)))
            map_e={int(g):'H'+str(i+1).zfill(2) for i,g in enumerate(order)}
            la=np.array([inp['maps'][space][int(g)] for g in ya]);le=np.array([map_e[int(g)] for g in ye])
            assignments[space+'_A_original']=la;assignments[space+'_E_sensibilidade']=le
            ctab=pd.crosstab(pd.Series(la,name='grupo_A'),pd.Series(le,name='grupo_E'))
            for ga in ctab.index:
                for ge in ctab.columns:
                    cross.append({'espaco':space,'grupo_A':ga,'grupo_E':ge,'n':int(ctab.loc[ga,ge]),
                                  'pct_no_grupo_A':float(100*ctab.loc[ga,ge]/ctab.loc[ga].sum())})
            base_sub=inp['frames']['subs'].query('espaco == @space and k == @k').copy()
            base_rot=inp['frames']['rot'].query('espaco == @space and k == @k').copy()
            expected=[int(mask.sum()) for mask in masks]
            check(expected==base_sub.sort_values('repeticao').n_condicoes_ajuste.astype(int).tolist(),
                  'Subamostragem nao reproduz tamanhos originais; conferir ordem/semente.')
            for row in base_sub.to_dict('records'):
                subs.append({'espaco':space,'entrada':'A','repeticao':int(row['repeticao']),
                    'ARI_todas':row['ARI_todas'],'ARI_compostos_fora':row['ARI_compostos_fora'],'origem':'etapa04_preservada'})
            for i,mask in enumerate(masks):
                m,_,_=fit(med['E'][mask,:4],space,k,seed+10000+i)
                y,_=fixed_predict(med['E'][:,:4],m)
                subs.append({'espaco':space,'entrada':'E','repeticao':i+1,'ARI_todas':adjusted_rand_score(ye,y),
                    'ARI_compostos_fora':adjusted_rand_score(ye[~mask],y[~mask]),'origem':'sensibilidade06'})
            for row in base_rot.to_dict('records'):
                rots.append({'espaco':space,'entrada':'A','replica_aplicacao':row['replica_aplicacao'],'ARI':row['ARI'],
                    'pct_mesmo_grupo':row['pct_mesmo_grupo'],'origem':'etapa04_preservada'})
            for r in range(3):
                train=np.median(ce[:,[i for i in range(3) if i!=r],:4],axis=1)
                m,yt,_=fit(train,space,k,seed+20000+r);yv,_=fixed_predict(ce[:,r,:4],m)
                rots.append({'espaco':space,'entrada':'E','replica_aplicacao':REPS[r],
                    'ARI':adjusted_rand_score(yt,yv),'pct_mesmo_grupo':float(100*np.mean(yt==yv)),
                    'origem':'sensibilidade06'})
            for variant,model,labels,ids,z,mp in [('A',original,la,ya,za,inp['maps'][space]),('E',model_e,le,ye,ze,map_e)]:
                cube=inp[variant]
                pred_reps=np.stack([fixed_predict(cube[:,i,:4],model)[0] for i in range(3)],axis=1)
                same=np.all(pred_reps==pred_reps[:,[0]],axis=1)
                assignments[space+'_'+variant+'_3rep_mesmo_grupo']=same
                for i,r in enumerate(REPS):assignments[space+'_'+variant+'_'+r]=[mp[int(g)] for g in pred_reps[:,i]]
                dmc=dm[[variant+'_'+b for b in CHANNELS]].to_numpy(float);yd,_=fixed_predict(dmc,model)
                dl=np.array([mp[int(g)] for g in yd])
                for ii,row in dm[[P,R,W]].iterrows():
                    dmrows.append(dict(row,espaco=space,entrada=variant,grupo=dl[ii]))
                sr=[x['ARI_todas'] for x in subs if x['espaco']==space and x['entrada']==variant]
                rr=[x['ARI'] for x in rots if x['espaco']==space and x['entrada']==variant]
                summaries.append({'espaco':space,'entrada':variant,'k_fixo':k,'n_condicoes':len(df),
                    'silhouette_interno':float(silhouette_score(z,ids)),
                    'ARI_subamostras_mediana':float(np.median(sr)),
                    'ARI_rotacoes_mediana':float(np.median(rr)), 'ARI_rotacoes_min':float(np.min(rr)),
                    'ARI_rotacoes_max':float(np.max(rr)),
                    'AMI_placa_tratamentos':float(adjusted_mutual_info_score(plate,ids)),
                    'AMI_placa_DMSO_projetados':float(adjusted_mutual_info_score(dm[P],yd)),
                    'pct_3rep_mesmo_grupo':float(100*same.mean()),
                    'eta2_magnitude_log1p_entrada':eta2(np.log1p(med[variant][:,:4]).mean(1),ids),
                    'ARI_entre_A_e_E':float(adjusted_rand_score(ya,ye))})
                for g in sorted(set(labels)):
                    ix=labels==g
                    for b,block in enumerate(BLOCKS):
                        for scale in ['A','E']:
                            v=med[scale][ix,b]
                            profiles.append({'espaco':space,'entrada_ajuste':variant,'grupo':g,'bloco':block,
                                'escala_exibicao':scale,'n':int(ix.sum()),'mediana':float(np.median(v)),
                                'q25':float(np.quantile(v,.25)),'q75':float(np.quantile(v,.75))})
                    for p in PLATES:
                        den=int(np.sum(plate==p));n=int(np.sum(ix&(plate==p)))
                        composition.append({'espaco':space,'entrada':variant,'grupo':g,'placa':p,'n':n,
                                            'n_placa':den,'pct_na_placa':100*n/den})
            print(space+': referencia preservada, sensibilidade ajustada e 3 rotacoes concluidas.',flush=True)
    # Diagnosticos continuos sem agrupar; todas as placas sao apresentadas.
    for sample in ['DMSO','tratamentos']:
        for variant in ['A','E']:
            for ir,r in enumerate(REPS):
                if sample=='DMSO':
                    part=dm.loc[dm[R]==r];raw=part[[variant+'_'+b for b in CHANNELS]].to_numpy(float);ps=part[P].to_numpy()
                else:raw=inp[variant][:,ir,:4];ps=plate
                cont=rep(raw,'contrastes_4canais')
                for p in PLATES:
                    for j,b in enumerate(CHANNELS):
                        vv=cont[ps==p,j]
                        effect.append({'amostra':sample,'entrada':variant,'replica':r,'placa':p,'bloco':b,
                                       'n':len(vv),'contraste_medio':float(vv.mean())})
    results={'resumo':pd.DataFrame(summaries),'subamostras':pd.DataFrame(subs),'rotacoes':pd.DataFrame(rots),
             'perfis':pd.DataFrame(profiles),'grupos_por_placa':pd.DataFrame(composition),'cruzamento_A_E':pd.DataFrame(cross),
             'contrastes_placa_replica':pd.DataFrame(effect),'projecao_DMSO':pd.DataFrame(dmrows),
             'atribuicoes':assignments,'conferencia_originais':inp['conferences']}
    results['localizacao_12_prioridades']=assignments.loc[assignments.prioridade_R1_R3_original].copy()
    for name,t in results.items():t.to_csv(out/'tabelas'/(name+'.csv'),index=False)
    (out/'modelos_sensibilidade_E.json').write_text(json.dumps(models,indent=2),encoding='utf-8')
    return results




## 3. Figuras
Seis figuras: concordancia entre replicas, associacao com placa, cruzamentos das particoes e perfis em RMS. As tabelas incluem tambem os controles projetados e todas as atribuicoes por replica.

In [ ]:
def figures(res,out):
    legends={};d=out/'figuras'
    def save(fig,name,caption):
        fig.tight_layout();fig.savefig(d/(name+'.png'),dpi=220,bbox_inches='tight')
        fig.savefig(d/(name+'.svg'),bbox_inches='tight');plt.close(fig);legends[name]=caption
    # Uma figura independente por diagnostico, sem sobrepor escalas diferentes.
    fig,ax=plt.subplots(figsize=(9,4.8))
    for i,space in enumerate(SPACES):
        for j,v in enumerate(['A','E']):
            t=res['rotacoes'].query('espaco == @space and entrada == @v').sort_values('replica_aplicacao')
            pos=i*3+j;ax.scatter(np.repeat(pos,3),t.ARI,s=45)
            ax.plot([pos-.22,pos+.22],[t.ARI.median()]*2,linewidth=2)
    ax.set_xticks([0,1,3,4],['Magnitude\nA original','Magnitude\nRMS sem P95','Contrastes\nA original','Contrastes\nRMS sem P95'])
    ax.set_ylabel('ARI entre ajuste em 2 replicas e aplicacao na terceira');ax.set_ylim(-.1,1.05)
    ax.set_title('Sensibilidade da concordancia entre replicas; k fixo')
    save(fig,'S01_estabilidade_replicas','Tres pontos por representacao. Segmentos: medianas. Nao sao intervalos de confianca; as rotacoes reutilizam as mesmas condicoes.')
    fig,ax=plt.subplots(figsize=(9,4.8));s=res['resumo'];xx=np.arange(len(s))
    ax.bar(xx,s.AMI_placa_tratamentos);ax.set_xticks(xx,[('Magnitude' if 'magnitude' in r.espaco else 'Contrastes')+'\n'+r.entrada for r in s.itertuples()])
    ax.set_ylabel('Informacao mutua ajustada (AMI) com placa');ax.set_title('Associacao descritiva entre grupo e placa')
    save(fig,'S02_associacao_placa','AMI nos tratamentos completos. Associacao com placa nao separa composicao quimica de efeitos tecnicos. Menor AMI nao prova melhor representacao biologica.')
    for num,(space,k) in enumerate(SPACES.items(),3):
        t=res['cruzamento_A_E'].query('espaco == @space').pivot(index='grupo_A',columns='grupo_E',values='n')
        fig,ax=plt.subplots(figsize=(7,4.5));ax.axis('off')
        table=ax.table(cellText=t.to_numpy(),rowLabels=t.index,colLabels=t.columns,cellLoc='center',loc='center')
        table.auto_set_font_size(False);table.set_fontsize(12);table.scale(1,2)
        ax.set_title(('Magnitude' if k==2 else 'Contrastes')+': cruzamento das particoes\nLinhas G: A original | Colunas H: RMS sem P95')
        save(fig,f'S{num:02d}_cruzamento_'+space,'Contagens, nao matriz de confusao. G e H pertencem a modelos distintos; numeros de grupo nao estabelecem equivalencia biologica.')
    for num,space in enumerate(SPACES,5):
        t=res['perfis'].query('espaco == @space and entrada_ajuste == "E" and escala_exibicao == "E" and bloco in @CHANNELS')
        fig,ax=plt.subplots(figsize=(9,4.8))
        for g in sorted(t.grupo.unique()):
            p=t.query('grupo == @g').set_index('bloco').loc[CHANNELS];y=p.mediana.to_numpy();
            ax.plot(range(4),y,marker='o',label=g+f' (n={p.n.iloc[0]})')
            ax.fill_between(range(4),p.q25,p.q75,alpha=.12)
        ax.set_xticks(range(4),CHANNELS);ax.set_ylabel('RMS pos-MAD, sem divisao pelo P95')
        ax.set_title('Perfis dos novos grupos H: '+('magnitude' if 'magnitude' in space else 'contrastes'));ax.legend()
        save(fig,f'S{num:02d}_perfis_RMS_'+space,'Medianas e quartis entre condicoes. Nao ha linha RMS=1 como limiar de atividade; A>1 continua restrito a regra original. RMS nao e fluorescencia bruta.')
    return legends




## 4. Exportacao e rastreabilidade
O plano e gravado antes dos novos ajustes. Os hashes das fontes sao conferidos antes e depois. Uma pasta nova recebe os resultados; os arquivos originais nao sao regravados. O rascunho do TCC exige revisao antes de incorporacao ao manuscrito.

In [ ]:
def report(res):
    lines=['# Sensibilidade dos agrupamentos a divisao pelo P95','',
      'Analise exploratoria pos-hoc. K fixo: 2 para magnitude; 4 para contrastes. Nenhuma busca de novo k.',
      'Comparam-se A=RMS/P95 e E=RMS pos-MAD. Os modelos A e seus resultados originais permanecem intactos.',
      'Os modelos E sao NOVOS ajustes de sensibilidade, nao uma correcao validada. Nenhum dado de R4 e lido.',
      '', '## Resultados condicionais']
    for space in SPACES:
        tab=res['resumo'].query('espaco == @space').set_index('entrada');a=tab.loc['A'];e=tab.loc['E']
        lines += ['', '### '+space,
          f'ARI entre particoes A e E: {e.ARI_entre_A_e_E:.4f}.',
          f'ARI mediano nas 3 rotacoes: A={a.ARI_rotacoes_mediana:.4f}; E={e.ARI_rotacoes_mediana:.4f}.',
          f'ARI mediano nas subamostras: A={a.ARI_subamostras_mediana:.4f}; E={e.ARI_subamostras_mediana:.4f}.',
          f'AMI com placa nos tratamentos: A={a.AMI_placa_tratamentos:.4f}; E={e.AMI_placa_tratamentos:.4f}.',
          f'Mesmo grupo nas 3 projecoes pelo modelo fixo: A={a.pct_3rep_mesmo_grupo:.2f}%; E={e.pct_3rep_mesmo_grupo:.2f}%.',
          f'Silhouette descritivo, em espacos diferentes: A={a.silhouette_interno:.4f}; E={e.silhouette_interno:.4f}.']
    lines += ['', '## Interpretacao e limites',
      'Retirar o P95 altera a geometria e reajusta a escala robusta e os centros na nova entrada; nao isola uma fracao causal de artefato.',
      'RMS ja depende de MAD, do conjunto de features e de suas covariancias. A omissao de P95 nao remove automaticamente efeitos de placa.',
      'K permaneceu fixo a partir da analise anterior; estes resultados nao otimizam a representacao E nem demonstram o numero verdadeiro de fenomenos.',
      'Silhouettes em geometrias diferentes nao sao desempenho preditivo comparavel. AMI menor com placa nao comprova recuperacao de biologia.',
      'Os 12 candidatos usam uma regra calculada em A. Sua localizacao nos grupos H e somente contexto, sem aplicar E>1 ou mudar status.',
      'DMSO recebem sempre o centro mais proximo. Suas atribuicoes nao sao hits, probabilidades ou falsos positivos.',
      'O k e todas as etapas anteriores foram definidos em R1-R3 completos; rotacoes nao constituem validacao independente de todo o pipeline.',
      'Se a concordancia entre replicas continuar baixa, registrar essa limitacao; nao selecionar algoritmos ou limiares ate melhorar o resultado.',
      'Encerrada esta comparacao, priorizar a sintese do TCC. Qualquer outro metodo requer uma nova pergunta e protocolo separado.']
    return '\n\n'.join(lines)


def run(root,out,environment='Local'):
    out=Path(out);check(not out.exists(),'Pasta de saida ja existe; use nova execucao.')
    inp=load(root)
    out.mkdir(parents=True);(out/'tabelas').mkdir();(out/'figuras').mkdir()
    plan={'natureza':'Sensibilidade exploratoria pos-hoc; nao nova validacao independente',
      'ambiente':environment,'utc':datetime.now(timezone.utc).isoformat(),'config':CFG,'fontes':SOURCES,
      'n_condicoes':len(inp['df']),'n_compostos_completos':int(inp['df'][E].nunique()),'n_dmso':len(inp['dmso']),
      'pergunta':'Como a omissao da divisao final por P95 modifica particoes, estabilidade e associacao com placa?',
      'variacao_permitida':'A=E/P95 versus E. Manter MAD, features, condicoes, quatro canais, k, algoritmo e protocolo de estabilidade.',
      'RMS_tratamentos':'Reconstrucao E=A*P95; referencia constante por placa/bloco, identidade conferida na etapa05.',
      'baseline':'Modelos, subamostras e rotacoes da etapa04 preservados, nao sobrescritos.',
      'interpretacao':'Nao usar E>1 como limiar nem procurar novos candidatos; grupos H nao equivalem aos G.',
      'criterio_encerramento':'Reportar ambos os resultados, inclusive falta de melhora, sem escolha de vencedor biologico.'}
    (out/'plano_antes_ajuste.json').write_text(json.dumps(plan,indent=2,ensure_ascii=False),encoding='utf-8')
    res=analyse(inp,out);caps=figures(res,out);txt=report(res)
    (out/'relatorio_sensibilidade.md').write_text(txt,encoding='utf-8')
    manuscript=['# Rascunho para revisao - sensibilidade a divisao pelo P95',
      'Foi realizada uma comparacao exploratoria pos-hoc entre os escores A=RMS/P95 e os RMS anteriores a essa divisao, mantendo a normalizacao MAD, as caracteristicas e as mesmas 2.447 condicoes. Os RMS dos tratamentos foram reconstruidos pela identidade E=A*P95, cuja referencia e constante por placa e bloco. Foram mantidos dois grupos na representacao de magnitude e quatro na de contrastes, sem nova selecao de k. Os modelos originais foram preservados e foram ajustados novos modelos apenas a representacao E, com escala robusta, 30 inicializacoes e o protocolo anterior de subamostragem e rotacao de replicas.']
    for space in SPACES:
        st=res['resumo'].query('espaco == @space').set_index('entrada');aa=st.loc['A'];ee=st.loc['E']
        manuscript.append(f'Na representacao {space}, a concordancia entre as particoes A e E foi ARI={ee.ARI_entre_A_e_E:.4f}. O ARI mediano de rotacao entre replicas passou de {aa.ARI_rotacoes_mediana:.4f} para {ee.ARI_rotacoes_mediana:.4f}; a associacao descritiva entre grupos e placas, avaliada por AMI, passou de {aa.AMI_placa_tratamentos:.4f} para {ee.AMI_placa_tratamentos:.4f}.')
    manuscript.append('A comparacao distingue sensibilidade a representacao de reprodutibilidade entre replicas. Diferencas de associacao com placa nao demonstram remocao causal de artefatos ou recuperacao de seletividade biologica. A analise permaneceu condicionada ao pre-processamento anterior e nao alterou os candidatos nem utilizou R4.')
    (out/'texto_TCC_sensibilidade.md').write_text('\n\n'.join(manuscript),encoding='utf-8')
    (out/'legendas.md').write_text('\n\n'.join('## '+k+'\n'+v for k,v in caps.items()),encoding='utf-8')
    (out/'fontes.md').write_text('\n'.join('- '+k+': '+s['drive_url'] for k,s in SOURCES.items())+'\n\nDocumentacao: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html\nhttps://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html\nhttps://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html\n',encoding='utf-8')
    for name,p in inp['paths'].items():check(sha256(p)==SOURCES[name]['sha256'],'Fonte alterada durante execucao.')
    manifest=dict(plan,n_figuras=len(caps),python=platform.python_version(),numpy=np.__version__,
       pandas=pd.__version__,sklearn=sklearn.__version__,leitores=inp['readers'],originais_preservados=True)
    manifest['saidas_sha256']={str(p.relative_to(out)):sha256(p) for p in sorted(out.rglob('*')) if p.is_file()}
    (out/'manifesto_sensibilidade.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    bundle=shutil.make_archive(str(out),'zip',root_dir=out)
    print(txt);print('\nCONCLUIDO: 6 figuras; comparacao A versus RMS; k fixo. Pasta:',out,'Pacote:',bundle)
    print('R4 nao lida. Modelos originais preservados. Novos modelos somente na analise de sensibilidade.')
    return res


In [ ]:
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=Path(os.environ['TCC_OUT']) if 'TCC_OUT' in os.environ else RAIZ/'resultados_tcc'/'sensibilidade_P95'/('R1_R3_'+RUN_ID)
RESULT=run(RAIZ,OUT,environment='Colab' if IN_COLAB else 'Local - verificacao separada')
display(RESULT['resumo'])

## 5. Revisao das figuras
ARI indica concordancia, nao acuracia. AMI indica associacao, nao a fracao de efeito tecnico. Uma diminuicao de associacao com placas sem melhora de reprodutibilidade nao constitui correcao validada.

In [ ]:
for fig_path in sorted((OUT/'figuras').glob('*.png')):
    display(Markdown('### '+fig_path.stem))
    display(Image(filename=str(fig_path),width=850))
display(Markdown((OUT/'relatorio_sensibilidade.md').read_text()))

## Decisao depois da execucao
Reportar ambas as representacoes, inclusive qualquer piora. Nao alterar thresholds, candidatos ou R4. Esta comparacao deve fechar a verificacao de sensibilidade desta linha de analise; o passo seguinte e sintetizar os resultados e revisar a narrativa, figuras e referencias do TCC.

Documentacao: [K-means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [escala robusta](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html), [ARI](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).